In [0]:

import dbutils

#  Define Volume paths
VOLUME_PATH = "/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/"

FILES = {
    "transactions": f"{VOLUME_PATH}/Transaction_Data_250k.csv",
    "cards":        f"{VOLUME_PATH}/Cards_Data.csv",
    "customers":    f"{VOLUME_PATH}/Cusmtomer_data.csv",
    "merchants":    f"{VOLUME_PATH}/merchant_table.csv",
}

BRONZE_TABLES = {
    "transactions": "bronze_transactions",
    "cards":        "bronze_cards",
    "customers":    "bronze_customers",
    "merchants":    "bronze_merchants",
}

In [0]:

# CELL 2: Verify files are visible from Databricks
files = (
    dbutils.fs.ls(VOLUME_PATH))
print(f" Files in {VOLUME_PATH}:")
for f in files:
    print(f"   {f.name}  ({f.size:,} bytes)")

# Quick peek at transactions
test_df = (spark.read.format("csv")
           .option("header", "true")
           .option("inferSchema", "true")
           .load(FILES["transactions"])
           .limit(3))

print("\n Sample rows from transactions:")
display(test_df.limit(5))

 Files in /Volumes/finance-fraud-pipeline/finance_schema/bank_raw/:
   Cards_Data.csv  (2,870,941 bytes)
   Cusmtomer_data.csv  (2,639,361 bytes)
   Transaction_Data_250k.csv  (43,513,673 bytes)
   merchant_table.csv  (38,336 bytes)

 Sample rows from transactions:


Transaction_ID,Customer_ID,Card_ID,Merchant_ID,Transaction_Date,Transaction_Time,Transaction_Amount,Payment_Method,Transaction_Channel,Device_Type,Transaction_Status,Is_International,Fraud_Flag,Fraud_Reason,Merchant_Risk_Level,Merchant_Category,Customer_State,Customer_City,Merchant_State,Merchant_City
TXN1000001,CUST100660,CARD500875,MER100380,2024-03-03,05:50:10,21748.02,Debit Card,Mobile App,Android Mobile,Successful,1,0,None,Low,Hotel,Tamil Nadu,Madurai,Tamil Nadu,Salem
TXN1000002,CUST111388,CARD514772,MER100017,2023-11-08,06:55:15,9960.6,Debit Card,POS,POS Terminal,Successful,0,0,None,Low,Travel,Kerala,Kollam,Haryana,Nuh
TXN1000003,CUST115872,CARD520625,MER100143,2023-01-11,02:17:28,838.61,Debit Card,ATM,ATM Machine,Successful,0,0,None,Low,Food Delivery,Uttar Pradesh,Agra,West Bengal,Kolkata


In [0]:
# ══════════════════════════════════════════════════════════
# CELL 3: Ingest all CSVs → Bronze Delta tables (UC-compatible)
# Why: Unity Catalog replaces input_file_name() with _metadata.file_path
# ══════════════════════════════════════════════════════════

from pyspark.sql.functions import current_timestamp, col

for name, path in FILES.items():
    table_name = BRONZE_TABLES[name]
    print(f"\n Ingesting '{name}' → {table_name}")

    # 1) Read CSV
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(path))

    # 2) Add tracking columns (UC-safe version)
    df = (df.withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", col("_metadata.file_path")))

    # 3) Write as Delta
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # 4) Report
    cnt = spark.table(table_name).count()
    print(f"    {table_name}: {cnt:,} rows")

print("\n Cell 3 done — Bronze layer complete")


 Ingesting 'transactions' → bronze_transactions
    bronze_transactions: 250,000 rows

 Ingesting 'cards' → bronze_cards
    bronze_cards: 32,457 rows

 Ingesting 'customers' → bronze_customers
    bronze_customers: 25,000 rows

 Ingesting 'merchants' → bronze_merchants
    bronze_merchants: 500 rows

 Cell 3 done — Bronze layer complete


In [0]:
# ══════════════════════════════════════════════════════════
# CELL 4: Verify all Bronze tables
# ══════════════════════════════════════════════════════════

print(" Bronze tables summary:")
print("=" * 50)
for name, table_name in BRONZE_TABLES.items():
    cnt = spark.table(table_name).count()
    print(f"   {table_name:<25} {cnt:>10,} rows")

print("\n Schema of bronze_transactions:")
spark.table("bronze_transactions").printSchema()

print("\n Preview:")
display(spark.table("bronze_transactions").limit(5))

 Bronze tables summary:
   bronze_transactions          250,000 rows
   bronze_cards                  32,457 rows
   bronze_customers              25,000 rows
   bronze_merchants                 500 rows

 Schema of bronze_transactions:
root
 |-- Transaction_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Card_ID: string (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)
 |-- Transaction_Time: time(6) (nullable = true)
 |-- Transaction_Amount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Transaction_Channel: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Transaction_Status: string (nullable = true)
 |-- Is_International: integer (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)
 |-- Fraud_Reason: string (nullable = true)
 |-- Merchant_Risk_Level: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Customer_St

Transaction_ID,Customer_ID,Card_ID,Merchant_ID,Transaction_Date,Transaction_Time,Transaction_Amount,Payment_Method,Transaction_Channel,Device_Type,Transaction_Status,Is_International,Fraud_Flag,Fraud_Reason,Merchant_Risk_Level,Merchant_Category,Customer_State,Customer_City,Merchant_State,Merchant_City,_ingested_at,_source_file
TXN1000001,CUST100660,CARD500875,MER100380,2024-03-03,05:50:10,21748.02,Debit Card,Mobile App,Android Mobile,Successful,1,0,None,Low,Hotel,Tamil Nadu,Madurai,Tamil Nadu,Salem,2026-10-13T10:05:26.977Z,dbfs:/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/Transaction_Data_250k.csv
TXN1000002,CUST111388,CARD514772,MER100017,2023-11-08,06:55:15,9960.6,Debit Card,POS,POS Terminal,Successful,0,0,None,Low,Travel,Kerala,Kollam,Haryana,Nuh,2026-10-13T10:05:26.977Z,dbfs:/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/Transaction_Data_250k.csv
TXN1000003,CUST115872,CARD520625,MER100143,2023-01-11,02:17:28,838.61,Debit Card,ATM,ATM Machine,Successful,0,0,None,Low,Food Delivery,Uttar Pradesh,Agra,West Bengal,Kolkata,2026-10-13T10:05:26.977Z,dbfs:/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/Transaction_Data_250k.csv
TXN1000004,CUST109049,CARD511773,MER100053,2023-05-25,01:48:56,52694.99,Credit Card,POS,POS Terminal,Successful,0,0,None,Low,Financial Services,Punjab,Ludhiana,Madhya Pradesh,Indore,2026-10-13T10:05:26.977Z,dbfs:/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/Transaction_Data_250k.csv
TXN1000005,CUST114459,CARD518771,MER100064,2023-10-22,00:23:32,1043.4,Debit Card,Online Web,Android Mobile,Successful,0,0,None,Low,Entertainment,Maharashtra,Thane,Madhya Pradesh,Jabalpur,2026-10-13T10:05:26.977Z,dbfs:/Volumes/finance-fraud-pipeline/finance_schema/bank_raw/Transaction_Data_250k.csv


In [0]:
# ══════════════════════════════════════════════════════════
# CELL 5: Verify Delta Lake features
# ══════════════════════════════════════════════════════════

print(" Delta history of bronze_transactions:")
spark.sql("DESCRIBE HISTORY bronze_transactions").show(truncate=False)

print("\n Reading bronze_transactions VERSION AS OF 0:")
spark.sql("SELECT COUNT(*) AS rows_at_v0 FROM bronze_transactions VERSION AS OF 0").show()

 Delta history of bronze_transactions:
+-------+-------------------+--------------+-----------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName         |operation                        |operationParameters                                                                                          